# 14. Scaffold split vs random drug split

**Question.** Is the model ranking (RF > CNN > graph) and the cold-drug performance level an accident of one random
drug split, or does it hold when chemically similar drugs are kept together?

**Design.** All 18,037 pancreatic pairs are pooled. Drugs are grouped by Bemis-Murcko scaffold (RDKit); whole scaffold
groups are held out as test (about 21% of drugs), so analogues of a test drug are not in training. Compared with a
random drug-grouped split of the same size, over 3 seeds each. Training size fixed at 12,929 pairs.

**Models.** RF + AE (300 trees) and cell-line-mean floor; same features and `y_clip` as elsewhere.
Drugs whose SMILES give no scaffold are assigned to their own group.

In [6]:
import numpy as np, pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr

ROOT = Path(r"D:\Pancreatic_GraphTCDR"); P = ROOT/"data"/"processed"; T = P/"target_v2"; RES = ROOT/"results"/"v2"
train = pd.read_csv(T/"train_v2.csv"); val = pd.read_csv(T/"val_v2.csv"); test = pd.read_csv(T/"test_v2.csv")
allp = pd.concat([train, val, test], ignore_index=True)
Z402 = np.load(P/"ae_embeddings_402x512.npy").astype(np.float32)
fp = np.load(P/"drug_morgan_fingerprints_1228x2048.npy").astype(np.float32)
smi = pd.read_csv(P/"drug_smiles_1228.csv")

def scaffold(s, i):
    try:
        m = Chem.MolFromSmiles(s)
        sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m) if m else ""
    except Exception:
        sc = ""
    return sc if sc else f"NOSCAF_{i}"          # acyclic / failed -> own group

smi["scaffold"] = [scaffold(s, i) for i, s in enumerate(smi["canonical_smiles"])]
drug_scaf = smi["scaffold"].values
print("drugs:", len(smi), "| unique scaffolds:", len(set(drug_scaf)),
      "| drugs with own-group (no scaffold):", int(smi.scaffold.str.startswith("NOSCAF").sum()))
allp["scaffold"] = drug_scaf[allp.drug_index.values]

drugs: 1228 | unique scaffolds: 983 | drugs with own-group (no scaffold): 42


In [7]:
from sklearn.model_selection import GroupShuffleSplit
def X_of(df): return np.hstack([Z402[df.omics_index.values], fp[df.drug_index.values]])
N_TRAIN = 12929
rows = []
for seed in [0, 1, 2]:
    for name, groups in [("random-drug", allp.drug_index.values), ("scaffold", allp.scaffold.values)]:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.21, random_state=seed)
        a, b = next(gss.split(allp, groups=groups))
        tr = allp.iloc[a].sample(min(N_TRAIN, len(a)), random_state=seed); te = allp.iloc[b]
        assert set(tr.drug_index).isdisjoint(set(te.drug_index))
        if name == "scaffold": assert set(tr.scaffold).isdisjoint(set(te.scaffold))
        cm = tr.groupby("omics_index").y_clip.mean(); gm = tr.y_clip.mean()
        floor = te.omics_index.map(cm).fillna(gm).values
        rf = RandomForestRegressor(300, max_features="sqrt", min_samples_leaf=3, random_state=seed, n_jobs=-1)
        rf.fit(X_of(tr), tr.y_clip.values); p = rf.predict(X_of(te)); y = te.y_clip.values
        rows.append(dict(split=name, seed=seed, n_test=len(te), test_drugs=te.drug_index.nunique(),
                         RF_R2=r2_score(y, p), RF_MAE=mean_absolute_error(y, p), RF_rho=spearmanr(y, p)[0],
                         floor_R2=r2_score(y, floor)))
        print(rows[-1])
res = pd.DataFrame(rows)
print(res.groupby("split")[["RF_R2", "RF_MAE", "RF_rho", "floor_R2"]].agg(["mean", "std"]).round(3))
res.to_csv(RES/"scaffold_split_comparison.csv", index=False)

{'split': 'random-drug', 'seed': 0, 'n_test': 3935, 'test_drugs': 258, 'RF_R2': 0.07600218568223593, 'RF_MAE': 0.5574480157752942, 'RF_rho': np.float64(0.35013387639481913), 'floor_R2': 0.004781443379669681}
{'split': 'scaffold', 'seed': 0, 'n_test': 3293, 'test_drugs': 224, 'RF_R2': 0.05540372911633318, 'RF_MAE': 0.5248747811411418, 'RF_rho': np.float64(0.3547121169210468), 'floor_R2': 0.014679062888851235}
{'split': 'random-drug', 'seed': 1, 'n_test': 3870, 'test_drugs': 258, 'RF_R2': 0.2186011397487756, 'RF_MAE': 0.5117328415690964, 'RF_rho': np.float64(0.37632622241464553), 'floor_R2': 0.002046369630376743}
{'split': 'scaffold', 'seed': 1, 'n_test': 3693, 'test_drugs': 232, 'RF_R2': 0.06893177501749725, 'RF_MAE': 0.5240102011228297, 'RF_rho': np.float64(0.3033317028381573), 'floor_R2': 0.011169847326679805}
{'split': 'random-drug', 'seed': 2, 'n_test': 3702, 'test_drugs': 258, 'RF_R2': 0.16367556781597736, 'RF_MAE': 0.5188649936372773, 'RF_rho': np.float64(0.4148909492201066), 'flo